In [6]:
# ============================================================
# Cell 1: Environment & Configuration
# ============================================================

from pathlib import Path

import ee
import geemap

# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

ADMIN_DIR = RAW_DIR / "admin"
CHIRPS_DIR = RAW_DIR / "chirps"

CHIRPS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Earth Engine
# ------------------------------------------------------------------

EE_PROJECT = "august-analyze"
DRIVE_FOLDER = "Zoryn"

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

# ------------------------------------------------------------------
# Pilot configuration
# ------------------------------------------------------------------

COUNTRY = "Kenya"
ADMIN1 = "Nairobi"

START_DATE = "2002-01-01"
END_DATE = "2024-12-31"

SCALE = 5566  # Native CHIRPS resolution (~0.05°)

DATASET = "UCSB-CHG/CHIRPS/DAILY"

EXPORT_DESCRIPTION = "zoryn_chirps_monthly_2002_2024"

OUTPUT_FILENAME = "chirps_monthly_2002_2024"

# ------------------------------------------------------------------
# Boundary
# ------------------------------------------------------------------

BOUNDARY = (
    ee.FeatureCollection("FAO/GAUL/2015/level1")
    .filter(ee.Filter.eq("ADM0_NAME", COUNTRY))
    .filter(ee.Filter.eq("ADM1_NAME", ADMIN1))
)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

print("=" * 60)
print("ZORYN CHIRPS EXPORT")
print("=" * 60)
print(f"Country     : {COUNTRY}")
print(f"Region      : {ADMIN1}")
print(f"Period      : {START_DATE} → {END_DATE}")
print(f"Dataset     : {DATASET}")
print(f"Scale (m)   : {SCALE}")
print(f"Output Dir  : {CHIRPS_DIR}")
print("=" * 60)

ZORYN CHIRPS EXPORT
Country     : Kenya
Region      : Nairobi
Period      : 2002-01-01 → 2024-12-31
Dataset     : UCSB-CHG/CHIRPS/DAILY
Scale (m)   : 5566
Output Dir  : c:\Projects\ews\zoryn\data\raw\chirps


In [2]:
# ============================================================
# Cell 2: Validate Nairobi boundary
# ============================================================

count = BOUNDARY.size().getInfo()

if count != 1:
    raise ValueError(
        f"Expected one Nairobi boundary, found {count}."
    )

feature = BOUNDARY.first()

properties = feature.toDictionary().getInfo()

print("=" * 60)
print("BOUNDARY")
print("=" * 60)
print(f"Features : {count}")
print(f"ADM0     : {properties['ADM0_NAME']}")
print(f"ADM1     : {properties['ADM1_NAME']}")
print("=" * 60)

BOUNDARY
Features : 1
ADM0     : Kenya
ADM1     : Nairobi


In [3]:
# ============================================================
# Cell 3: Build monthly CHIRPS collection
# ============================================================

from datetime import datetime

collection = (
    ee.ImageCollection(DATASET)
    .filterDate(START_DATE, END_DATE)
)

start = datetime.strptime(START_DATE, "%Y-%m-%d")
end = datetime.strptime(END_DATE, "%Y-%m-%d")

months = (
    (end.year - start.year) * 12
    + end.month
    - start.month
    + 1
)

monthly = []

for i in range(months):

    current = ee.Date(START_DATE).advance(i, "month")

    image = (
        collection
        .filterDate(
            current,
            current.advance(1, "month"),
        )
        .sum()
        .rename("precipitation")
        .set(
            {
                "system:index": current.format("YYYYMM"),
                "timestamp": current.format("YYYY-MM"),
            }
        )
    )

    monthly.append(image)

monthly = ee.ImageCollection(monthly)

print("=" * 60)
print("MONTHLY COLLECTION")
print("=" * 60)
print(f"Images : {monthly.size().getInfo():,}")
print("=" * 60)

MONTHLY COLLECTION
Images : 276


In [4]:
# ============================================================
# Cell 4: Convert monthly collection to multiband image
# ============================================================

# ------------------------------------------------------------------
# Rename each image using YYYYMM
# ------------------------------------------------------------------

def rename_band(image):
    band_name = ee.String(image.get("system:index"))

    return image.rename(band_name)


monthly = monthly.map(rename_band)

# ------------------------------------------------------------------
# Convert ImageCollection to multiband image
# ------------------------------------------------------------------

multiband = monthly.toBands()

# ------------------------------------------------------------------
# Remove the collection prefix added by toBands()
# ------------------------------------------------------------------

band_names = monthly.aggregate_array("system:index")

multiband = multiband.rename(band_names)

# ------------------------------------------------------------------
# Clip to Nairobi
# ------------------------------------------------------------------

multiband = multiband.clip(BOUNDARY)

# ------------------------------------------------------------------
# Summary
# ------------------------------------------------------------------

bands = multiband.bandNames().getInfo()

print("=" * 60)
print("MULTIBAND IMAGE")
print("=" * 60)
print(f"Bands : {len(bands):,}")
print(f"First : {bands[0]}")
print(f"Last  : {bands[-1]}")
print("=" * 60)

MULTIBAND IMAGE
Bands : 276
First : 200201
Last  : 202412


In [5]:
# ============================================================
# Cell 5: Export multiband CHIRPS GeoTIFF
# ============================================================

region = BOUNDARY.geometry()

task = ee.batch.Export.image.toDrive(
    image=multiband,
    description=EXPORT_DESCRIPTION,
    folder="Zoryn",
    fileNamePrefix=OUTPUT_FILENAME,
    region=region,
    scale=SCALE,
    crs="EPSG:4326",
    maxPixels=1e13,
    fileFormat="GeoTIFF",
)

task.start()

print("=" * 60)
print("EXPORT STARTED")
print("=" * 60)
print(f"Description : {EXPORT_DESCRIPTION}")
print(f"Filename    : {OUTPUT_FILENAME}.tif")
print(f"Folder      : Google Drive/Zoryn")
print("=" * 60)

EXPORT STARTED
Description : zoryn_chirps_monthly_2002_2024
Filename    : chirps_monthly_2002_2024.tif
Folder      : Google Drive/Zoryn


In [7]:
# ============================================================
# Cell 5: Export multiband CHIRPS GeoTIFF
# ============================================================

region = BOUNDARY.geometry()

task = ee.batch.Export.image.toDrive(
    image=multiband,
    description=EXPORT_DESCRIPTION,
    folder=DRIVE_FOLDER,
    fileNamePrefix=OUTPUT_FILENAME,
    region=region,
    scale=SCALE,
    crs="EPSG:4326",
    maxPixels=1e13,
    fileFormat="GeoTIFF",
)

task.start()

print("=" * 60)
print("EXPORT STARTED")
print("=" * 60)
print(f"Description : {EXPORT_DESCRIPTION}")
print(f"Drive Folder: {DRIVE_FOLDER}")
print(f"Filename    : {OUTPUT_FILENAME}.tif")
print("=" * 60)

EXPORT STARTED
Description : zoryn_chirps_monthly_2002_2024
Drive Folder: Zoryn
Filename    : chirps_monthly_2002_2024.tif


In [8]:
# ============================================================
# Cell 6: Monitor export
# ============================================================

import time

print("=" * 60)
print("MONITORING EXPORT")
print("=" * 60)

while True:

    status = task.status()

    state = status["state"]

    print(f"Status : {state}")

    if state == "COMPLETED":

        print("\nExport completed successfully.")

        print(
            "\nMove the downloaded file to:\n"
            "data/raw/chirps/\n"
            "and rename it to:\n"
            "chirps_monthly_2002_2024.tif"
        )

        break

    elif state in ("FAILED", "CANCELLED"):

        print("\nExport did not complete.")

        print(status)

        break

    time.sleep(30)

print("=" * 60)

MONITORING EXPORT
Status : RUNNING
Status : RUNNING
Status : COMPLETED

Export completed successfully.

Move the downloaded file to:
data/raw/chirps/
and rename it to:
chirps_monthly_2002_2024.tif
